# Copa América 2024 — the citywide visitor lift

**Question:** when a summer soccer tournament with travelling fans comes to our stadiums, how much
extra card spend shows up across the whole host city, by shop type, and how does it move day by day?

**Method (difference-in-differences):** for each city take spend on each day of the tournament window
(20 Jun – 14 Jul 2024) divided by the same calendar day in 2023, using only shops that traded in both
summers so shops joining or leaving the panel cancel out. Eight of our cities hosted matches
(Atlanta, Dallas, Houston, Kansas City, Los Angeles, Miami, New York/New Jersey, San Francisco).
Three did not (Boston, Philadelphia, Seattle). Host ratio ÷ non-host ratio = the lift. Multiplicative
noise cancels in every ratio here.

Inputs: `spend-patterns-rice.parquet` in the Drive cache, plus `copa_america_2024.csv` and
`stadiums.csv` from `data/worldcup/` (upload when asked).

In [ ]:
# ============================================================
# STEP 0 — mount Drive, inputs
# ============================================================
import os, json
import numpy as np, pandas as pd
pd.set_option("display.width", 220)

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/ricehack_cache"
except ImportError:
    CACHE = os.path.expanduser("~/ricehack_cache")

MUST_HAVE = {"copa_america_2024.csv": {"date","stadium","market"}, "stadiums.csv": {"stadium","market","lat","lon"},
             "nfl_home_games.csv": {"date","stadium","market"}}
def need(name):
    p = f"{CACHE}/{name}"
    while True:
        if os.path.exists(p) and os.path.getsize(p) > 0:
            try:
                t = pd.read_csv(p)
                if MUST_HAVE[name].issubset(t.columns): return t
            except Exception: pass
            os.remove(p); print(f"{name} in the cache was empty or the wrong file — removed it")
        print(f"upload {name} from data/worldcup/ in the repo (any filename is fine)")
        up = files.upload()
        open(p, "wb").write(list(up.values())[0])

copa     = need("copa_america_2024.csv"); copa["date"] = pd.to_datetime(copa["date"])
stadiums = need("stadiums.csv")
copa = copa[copa.in_our_cities == 1]
HOST = sorted(copa.market.unique())
ALLC = ["Atlanta","Boston","Dallas","Houston","Kansas City","Los Angeles","Miami",
        "New York/New Jersey","Philadelphia","San Francisco Bay Area","Seattle"]
CTRL = [c for c in ALLC if c not in HOST]
print("host:", HOST); print("control:", CTRL)

T0, T1 = pd.Timestamp("2024-06-20"), pd.Timestamp("2024-07-14")   # tournament window
P0, P1 = pd.Timestamp("2024-05-16"), pd.Timestamp("2024-06-12")   # pre-window: should show NO lift


In [ ]:
# ============================================================
# STEP 1 — load EFW shops in the 11 cities, keep only shops present in BOTH summers
# ============================================================
CENTRES = {"Atlanta":(33.749,-84.388),"Boston":(42.360,-71.058),"Dallas":(32.777,-96.797),"Houston":(29.760,-95.370),
           "Kansas City":(39.100,-94.579),"Los Angeles":(34.052,-118.244),"San Francisco Bay Area":(37.775,-122.419),
           "Miami":(25.775,-80.194),"New York/New Jersey":(40.713,-74.006),"Philadelphia":(39.953,-75.165),"Seattle":(47.606,-122.332)}
RADIUS_KM = 75
EFW_NAICS3 = {"722","445","447","721","711","712","713","562","221","311","312","424","485","481","488"}
def layer_of(n3):
    if n3 in {"722","445","311","312","424"}: return "Food"
    if n3 in {"447","221"}:                   return "Energy"
    if n3 in {"721","562"}:                   return "Water"
    if n3 in {"711","712","713"}:             return "Venue"
    return "Other_EFW"

COLS = ["PLACEKEY","MARKET","LATITUDE","LONGITUDE","NAICS_CODE","SPEND_DATE_RANGE_START","SPEND_BY_DAY"]
df = pd.read_parquet(f"{CACHE}/spend-patterns-rice.parquet", columns=COLS)
df["n3"] = df.NAICS_CODE.astype(str).str[:3]
df = df[df.n3.isin(EFW_NAICS3) & df.MARKET.isin(ALLC)].copy()
df["layer"] = df.n3.map(layer_of)
df["LATITUDE"]  = pd.to_numeric(df.LATITUDE,  errors="coerce"); df["LONGITUDE"] = pd.to_numeric(df.LONGITUDE, errors="coerce")
df = df.dropna(subset=["LATITUDE","LONGITUDE"])
lat0 = df.MARKET.map(lambda m: CENTRES[m][0]); lon0 = df.MARKET.map(lambda m: CENTRES[m][1])
d_km = np.sqrt(((df.LATITUDE-lat0)*111)**2 + ((df.LONGITUDE-lon0)*111*np.cos(np.radians(lat0)))**2)
df = df[d_km <= RADIUS_KM].copy()
df["month"] = pd.to_datetime(df.SPEND_DATE_RANGE_START).dt.strftime("%Y-%m")

# balanced panel: a shop must have May-Jul in BOTH 2023 and 2024
need_m = {"2023-05","2023-06","2023-07","2024-05","2024-06","2024-07"}
have = df.groupby("PLACEKEY").month.agg(lambda s: need_m.issubset(set(s)))
keep_keys = have[have].index
df = df[df.PLACEKEY.isin(keep_keys) & df.month.str[:4].isin(["2023","2024"])].copy()
print(f"{df.PLACEKEY.nunique():,} shops present in both summers, {len(df):,} shop-months")
print(df.groupby("MARKET").PLACEKEY.nunique().to_string())


In [ ]:
# ============================================================
# STEP 2 — daily citywide spend per market x layer, both years
# ============================================================
def as_vals(v):
    if isinstance(v, dict): return [v[k] for k in sorted(v)]
    if isinstance(v, (list, np.ndarray)): return list(v)
    if isinstance(v, str):
        try: p = json.loads(v)
        except Exception: return []
        return [p[k] for k in sorted(p)] if isinstance(p, dict) else list(p)
    return []
s_arr = df.SPEND_BY_DAY.map(as_vals); n = s_arr.map(len).values; ok = n > 0
df, s_arr, n = df[ok], s_arr[ok], n[ok]
daily = pd.DataFrame({
    "MARKET": np.repeat(df.MARKET.values, n),
    "layer":  np.repeat(df.layer.values, n),
    "PLACEKEY": np.repeat(df.PLACEKEY.values, n),
    "date":   np.concatenate([pd.date_range(s, periods=k, freq="D").values for s, k in zip(pd.to_datetime(df.SPEND_DATE_RANGE_START), n)]),
    "spend":  np.concatenate(s_arr.values).astype("float64"),
})
city = daily.groupby(["MARKET","layer","date"], as_index=False).spend.sum()
allrow = city.groupby(["MARKET","date"], as_index=False).spend.sum(); allrow["layer"] = "All"
city = pd.concat([city, allrow], ignore_index=True)
city.to_parquet(f"{CACHE}/copa_city_daily_balanced.parquet", index=False)
print(f"{len(city):,} market x layer x day rows")


In [ ]:
# ============================================================
# STEP 3 — window ratios: 2024 / 2023, host vs control
# ============================================================
def window_sum(d0, d1, year_shift=0):
    a, b = d0 - pd.DateOffset(years=year_shift), d1 - pd.DateOffset(years=year_shift)
    w = city[(city.date >= a) & (city.date <= b)]
    return w.groupby(["MARKET","layer"]).spend.sum()

def ratio_table(d0, d1):
    r = (window_sum(d0, d1, 0) / window_sum(d0, d1, 1)).unstack("layer")
    r["group"] = np.where(r.index.isin(HOST), "host", "control")
    return r

def summarise(r, title):
    print(f"\n=== {title} ===")
    print("per city: spend in window 2024 / same dates 2023, balanced panel (1.00 = no change)")
    print(r.round(3).to_string())
    pooled = {}
    for lyr in [c for c in r.columns if c != "group"]:
        h = r.loc[r.group == "host", lyr].median(); c = r.loc[r.group == "control", lyr].median()
        pooled[lyr] = dict(host_median=h, control_median=c, lift=h / c)
    out = pd.DataFrame(pooled).T
    print("\nhost median ÷ control median = LIFT (this is the number the forecast wants)")
    print(out.round(3).to_string())
    return out

pre  = summarise(ratio_table(P0, P1), "A. PRE-window 16 May – 12 Jun: lift here should be ~1.00 (no tournament yet)")
tour = summarise(ratio_table(T0, T1), "B. TOURNAMENT window 20 Jun – 14 Jul: the answer")
tour.to_csv(f"{CACHE}/copa_citywide_lift_by_layer.csv")


In [ ]:
# ============================================================
# STEP 4 — day by day: does the lift follow the matches, or sit flat all tournament?
# ============================================================
def double_ratio(layer="All", smooth=7):
    w = city[city.layer == layer].pivot_table(index="date", columns="MARKET", values="spend", fill_value=0.0)
    w = w.rolling(smooth, center=True, min_periods=1).mean()
    y24 = w.loc["2024-05-01":"2024-08-15"]; y23 = w.loc["2023-05-01":"2023-08-15"]
    y23.index = y23.index + pd.DateOffset(years=1)          # align calendar days
    yoy = (y24 / y23.reindex(y24.index))
    host = yoy[HOST].median(axis=1); ctrl = yoy[CTRL].median(axis=1)
    return (host / ctrl).rename("lift"), yoy

lift_all, yoy = double_ratio("All")
print("=== C. Citywide lift, all EFW shops, host ÷ control, 7-day smoothed, one row per week ===")
print(lift_all.resample("W").mean().round(3).to_string())

print("\n=== D. Same, by shop type, tournament weeks only ===")
tab = pd.DataFrame({l: double_ratio(l)[0] for l in ["Food","Energy","Water","Venue"]})
print(tab.loc[T0:T1].resample("W").mean().round(3).to_string())

# does a city's OWN match day add anything on top of the tournament-wide lift?
print("\n=== E. Host cities: day-offset from their own match, citywide yoy ratio ÷ same city's tournament-window average ===")
rows = []
for m in copa.itertuples():
    base = yoy.loc[T0:T1, m.market].median()
    for k in range(-3, 4):
        d = m.date + pd.Timedelta(days=k)
        if d in yoy.index: rows.append(dict(offset=k, rel=yoy.at[d, m.market] / base))
E = pd.DataFrame(rows).groupby("offset").rel.agg(["median","count"]).round(3)
print(E.to_string())
print("^ 1.00 = a match day is no different from any other tournament day citywide.")


In [ ]:
# ============================================================
# STEP 5 — stadium-block spike, same ring method as the NFL notebook, on the 19 Copa matches
# ============================================================
RINGS = [(0,2),(2,5),(5,10),(10,25)]
st = stadiums.set_index("market")
sub = df[df.MARKET.isin(HOST)].copy()
la0 = sub.MARKET.map(st.lat); lo0 = sub.MARKET.map(st.lon)
sub["km"] = np.sqrt(((sub.LATITUDE-la0)*111)**2 + ((sub.LONGITUDE-lo0)*111*np.cos(np.radians(la0)))**2)
sub = sub[sub.km <= 25]
sub["ring"] = pd.cut(sub.km, [0,2,5,10,25], labels=[f"{a}-{b} km" for a,b in RINGS], include_lowest=True).astype(str)
ring_daily = daily[daily.PLACEKEY.isin(sub.PLACEKEY)].merge(sub[["PLACEKEY","ring"]].drop_duplicates(), on="PLACEKEY")
ring_daily = ring_daily.groupby(["MARKET","ring","date"], as_index=False).spend.sum()
wide = ring_daily.pivot_table(index=["MARKET","ring"], columns="date", values="spend", fill_value=0.0)

contam = {(m.market, (m.date + pd.Timedelta(days=k)).normalize()) for m in copa.itertuples() for k in range(-3,4)}
rows = []
for m in copa.itertuples():
    for k in range(-3, 4):
        d = (m.date + pd.Timedelta(days=k)).normalize()
        if d not in wide.columns: continue
        ctrl_dates = [c for c in [d + pd.Timedelta(weeks=w) for w in (-4,-3,-2,-1,1,2,3,4)] if c in wide.columns and (m.market, c) not in contam]
        if len(ctrl_dates) < 3: continue
        s = wide.loc[m.market]; obs = s[d]; ctrl = s[ctrl_dates].median(axis=1); okk = ctrl > 0
        for ring, o, c in zip(s.index[okk], obs[okk], ctrl[okk]):
            rows.append(dict(market=m.market, date=m.date, offset=k, ring=ring, obs=o, ctrl=c))
R = pd.DataFrame(rows)
R["bucket"] = pd.cut(R.offset, [-4,-2,1,3], labels=["before (-3,-2)","around (-1,0,+1)","after (+2,+3)"])
print("=== F. Copa matches: pooled sum(observed)/sum(control) by ring and bucket — compare with NFL Table C ===")
print(R.groupby(["ring","bucket"], observed=True).apply(lambda x: x.obs.sum()/x.ctrl.sum(), include_groups=False).unstack("bucket").round(3).to_string())
print("\nshops per ring in host cities:"); print(sub.groupby(["MARKET","ring"]).PLACEKEY.nunique().unstack().fillna(0).astype(int).to_string())


## How to read what comes back

**Table A first.** It is the same calculation on the four weeks *before* the tournament. Every LIFT
should sit near 1.00. If host cities were already running 10% hotter than Boston/Philly/Seattle in
May, then Table B is not the tournament, it's something else (weather, a Taylor Swift tour, panel quirks),
and we cannot use it.

**Table B is the citywide number.** Read the `lift` column. `All` around 1.05 means a host city's
EFW shops did about 5% more during the tournament than they would have without it. `Water` is
lodging: if it moves, people stayed over, which is the thing NFL games could not show. `Food` is
restaurants and grocery. Compare `All` with the Club World Cup's published +7%: same neighbourhood
means we're not crazy.

**Table C** shows the lift week by week from May to mid-August. You want to see it near 1.00
before 20 June, rise during the tournament, and settle back after 14 July. If it never comes back
down, part of what you're seeing is not the tournament.

**Table D** is the same by shop type, tournament weeks only.

**Table E answers the "does the match day itself matter citywide" question.** If offsets 0 and +1
are near 1.00, a city's own match adds nothing citywide beyond the tournament-wide lift, and the
model should carry the visitor effect as a flat lift over the whole tournament (scaled by how long
a city stays in the tournament), plus only the stadium-block spike from Table F.

**Table F** repeats the NFL ring analysis on Copa's 19 matches. If `0-2 km` `around` lands near
the NFL's 1.33, the stadium-block spike is confirmed from two independent event types.

**Caveats to keep in mind:** the control is three cities, so a single odd summer in Seattle
moves the median; Copa crowds and travelling-fan numbers were smaller than a World Cup, so
Table B is a floor; and Table B is a summer-to-summer ratio, so it also nets out anything else
that changed 2023→2024 in host cities but not control cities.
